In [1]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.neural_network import MLPRegressor

In [2]:
FILE = 'games-players-processed/games_training_set_10.csv'

### Leitura

In [3]:
df = pd.read_csv(FILE)

C:\Users\duilio\AppData\Local\Temp\ipykernel_12264\1324869264.py:1: DtypeWarning: Columns (4,44,88,130,174,216,260,302,344,386,428,468,512,552,594,638,682,724,766,808,850) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(FILE)


In [4]:
df = df.select_dtypes(exclude=['category', 'object'])
df = df.fillna(0)
df = df.T.drop_duplicates().T # Transpose, drop duplicate 'rows' (original columns), and transpose back
df = df.sort_values(by='startTimestamp_y', ascending=True)

In [5]:
df

,gameId_p1_home_x,startTimestamp_p1_home_x,tournamentId_p1_home_x,round_p1_home_x,hasPlayerStatistics_p1_home_x,homeTeamId_p1_home_x,homeTeamFoundationDateTimestamp_p1_home_x,homeScoreNormalTime_p1_home_x,homeScorePeriod1_p1_home_x,awayTeamId_p1_home_x,...,homeExpectedgoals_p8_home_x,awayExpectedgoals_p8_home_x,homeExpectedgoals_p9_home_x,awayExpectedgoals_p9_home_x,homeExpectedgoals_p9_away_x,awayExpectedgoals_p9_away_x,homeExpectedgoals_p10_away_x,awayExpectedgoals_p10_away_x,homeExpectedgoals_p10_home_x,awayExpectedgoals_p10_home_x
2702,6698531,1444867200,83,30,True,1963,-1746748800.0,0,0.0,1969,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2701,6698537,1444870800,83,30,True,1967,-1444348800.0,2,1.0,1954,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2704,6698527,1444867200,83,30,True,1985,-1532304000.0,3,1.0,5981,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2703,6698537,1444870800,83,30,True,1967,-1444348800.0,2,1.0,1954,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2705,6698523,1444953600,83,30,True,5926,-2092176000.0,1,1.0,1968,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2011,15235530,1777161600,83,13,True,1981,-1260230400.0,1,0.0,21982,...,1.08,0.53,0.61,0.8,1.65,0.85,1.95,0.47,1.53,0.57
2010,15235533,1777246200,83,13,True,1977,-1949356800.0,0,0.0,5981,...,0.91,0.65,0.75,1.38,1.12,1.37,1.5,0.71,1.24,0.57
2013,15235529,1777152600,83,13,True,1958,-2382652800.0,2,0.0,1966,...,1.17,1.24,1.65,0.85,2.11,0.54,0.85,2.22,0.76,1.81
2012,15235527,1777246200,83,13,True,1961,-2128550400.0,2,0.0,21845,...,0.81,1.26,1.15,0.72,0.61,0.8,1.56,1.66,1.53,0.57


In [26]:
train_size = int(len(df) * 0.8)
train_data = df.iloc[:train_size]
test_data = df.iloc[train_size:]

X_train = train_data.filter(regex='_x$')
X_test = test_data.filter(regex='_x$')
X_train = train_data.filter(regex='^(homeScoreNormalTime|awayScoreNormalTime).*_x$')
X_test = test_data.filter(regex='^(homeScoreNormalTime|awayScoreNormalTime).*_x$')
print(X_train.columns)

#y_train = train_data['homePasses_y'] + train_data['awayPasses_y']
#y_test = test_data['homePasses_y'] + test_data['awayPasses_y']
#y_train = train_data['homeCornerkicks_y'] + train_data['awayCornerkicks_y']
#y_test = test_data['homeCornerkicks_y'] + test_data['awayCornerkicks_y']
y_train = (train_data['homeScoreNormalTime_y'] - train_data['awayScoreNormalTime_y'])
y_test = (test_data['homeScoreNormalTime_y'] - test_data['awayScoreNormalTime_y'])
scaler = StandardScaler().set_output(transform="pandas")
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Index(['homeScoreNormalTime_p1_home_x', 'awayScoreNormalTime_p1_home_x',
       'homeScoreNormalTime_p2_home_x', 'awayScoreNormalTime_p2_home_x',
       'homeScoreNormalTime_p3_home_x', 'awayScoreNormalTime_p3_home_x',
       'homeScoreNormalTime_p4_home_x', 'awayScoreNormalTime_p4_home_x',
       'homeScoreNormalTime_p5_home_x', 'awayScoreNormalTime_p5_home_x',
       'homeScoreNormalTime_p6_home_x', 'awayScoreNormalTime_p6_home_x',
       'homeScoreNormalTime_p7_home_x', 'awayScoreNormalTime_p7_home_x',
       'homeScoreNormalTime_p8_home_x', 'awayScoreNormalTime_p8_home_x',
       'homeScoreNormalTime_p9_home_x', 'awayScoreNormalTime_p9_home_x',
       'homeScoreNormalTime_p10_home_x', 'awayScoreNormalTime_p10_home_x',
       'homeScoreNormalTime_p1_away_x', 'awayScoreNormalTime_p1_away_x',
       'homeScoreNormalTime_p2_away_x', 'awayScoreNormalTime_p2_away_x',
       'homeScoreNormalTime_p3_away_x', 'awayScoreNormalTime_p3_away_x',
       'homeScoreNormalTime_p4_away_x', 'awayScor

In [30]:
model = RandomForestRegressor(n_estimators=100, max_features=None, random_state=42, verbose=1, n_jobs=8)
model.fit(X_train, y_train)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished


RandomForestRegressor(max_features=None, n_jobs=8, random_state=42, verbose=1)

In [11]:
model = MLPRegressor(hidden_layer_sizes=(400, 200, 100, 50, 25, 12), activation='relu', solver='adam', alpha=0.001, early_stopping=True, tol=0.00001, max_iter=2000, verbose=1)
model.fit(X_train, y_train)

Iteration 1, loss = 1.18485337
Validation score: -0.019490
Iteration 2, loss = 1.04515431
Validation score: -0.004421
Iteration 3, loss = 0.96108355
Validation score: -0.015884
Iteration 4, loss = 0.83761118
Validation score: -0.098407
Iteration 5, loss = 0.64899508
Validation score: -0.215227
Iteration 6, loss = 0.38712919
Validation score: -0.331587
Iteration 7, loss = 0.20920205
Validation score: -0.327306
Iteration 8, loss = 0.14056630
Validation score: -0.276609
Iteration 9, loss = 0.09627984
Validation score: -0.308115
Iteration 10, loss = 0.06901066
Validation score: -0.333016
Iteration 11, loss = 0.06588072
Validation score: -0.312654
Iteration 12, loss = 0.05319499
Validation score: -0.293171
Iteration 13, loss = 0.05685840
Validation score: -0.263789
Validation score did not improve more than tol=0.000010 for 10 consecutive epochs. Stopping.


MLPRegressor(alpha=0.001, early_stopping=True,
             hidden_layer_sizes=(400, 200, 100, 50, 25, 12), max_iter=2000,
             tol=1e-05, verbose=1)

In [31]:
# 5. Make predictions
y_pred = model.predict(X_test)

# 6. Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(mse, r2)

print(y_test, y_pred)

2.5210130071599046 -0.020155871885711107
1253     1
1255     2
1258    -1
1256     0
1257     1
        ..
2011     0
2010     0
2013     2
2012    -1
2014     1
Length: 838, dtype: object [ 0.17  0.22  0.37  0.32  0.88 -0.1   0.41  0.24  0.47  0.31 -0.26  0.56
  0.21  0.61  0.19  0.1   0.38  0.51  0.09  0.77  0.36  0.19  0.28  0.53
  0.24  0.28  0.71  0.36  0.39  0.07  0.69  0.36  0.19  0.2  -0.07 -0.05
  0.68  0.74  0.16  0.12  0.44  0.59  0.78  0.26  0.52  0.64  1.06  0.22
  0.73  0.5   0.53  0.55  0.53  0.08  0.34  0.34  0.46  0.53  0.36  0.76
  0.67  1.04  0.7   0.04  0.82  0.46  0.79  0.27  0.44  0.18  0.72  0.38
  0.61  0.7   0.21  0.28  0.26  0.43  0.2   0.39  0.54  0.68  0.11  0.41
  0.47  0.6   0.92  0.03  0.77  0.7   0.42  0.28  0.12  0.19  0.49  0.4
  0.49  0.3   0.75  0.43  0.38  0.3   0.44  0.85  0.05  0.56  0.55  0.25
  0.31  0.27  0.61  0.41  0.57 -0.01  0.4   0.56  0.66  0.45  0.59  0.38
  0.57  0.26  0.42  0.26  0.14  0.21  0.67  0.35  0.99  0.69  0.41  1.05
  0.37  0

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished


In [29]:
# Assume 'model' is your trained model and 'X' is your training data
importances = model.feature_importances_
feature_names = X_train.columns

# Create a DataFrame for easy sorting
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})

# Sort and print
ordered_imp = feature_imp_df.sort_values(by='Importance', ascending=False)
print(ordered_imp[:50])

                           Feature  Importance
36   homeScoreNormalTime_p9_away_x    0.029614
34   homeScoreNormalTime_p8_away_x    0.029263
12   homeScoreNormalTime_p7_home_x    0.028731
24   homeScoreNormalTime_p3_away_x    0.028464
30   homeScoreNormalTime_p6_away_x    0.028316
26   homeScoreNormalTime_p4_away_x    0.028021
20   homeScoreNormalTime_p1_away_x    0.027614
18  homeScoreNormalTime_p10_home_x    0.027467
32   homeScoreNormalTime_p7_away_x    0.027343
2    homeScoreNormalTime_p2_home_x    0.027069
14   homeScoreNormalTime_p8_home_x    0.026920
38  homeScoreNormalTime_p10_away_x    0.026824
9    awayScoreNormalTime_p5_home_x    0.026735
4    homeScoreNormalTime_p3_home_x    0.026438
6    homeScoreNormalTime_p4_home_x    0.026421
35   awayScoreNormalTime_p8_away_x    0.026412
28   homeScoreNormalTime_p5_away_x    0.026273
16   homeScoreNormalTime_p9_home_x    0.026161
22   homeScoreNormalTime_p2_away_x    0.026157
39  awayScoreNormalTime_p10_away_x    0.025583
8    homeScor

In [20]:
rf = RandomForestRegressor()
random_grid = {
    'n_estimators': [1000, 2000, 3000],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}
rf_random = GridSearchCV(estimator=rf, param_grid=random_grid, verbose=3, n_jobs=8)
rf_random.fit(X_train, y_train)
print(rf_random.best_params_)

Fitting 5 folds for each of 324 candidates, totalling 1620 fits


KeyboardInterrupt: 